In [5]:
from langchain_community.vectorstores import Chroma


In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
load_dotenv()
#--------------------------------------------------------------------
loader=PyPDFLoader('Attention_all_you_need.pdf')
documents= loader.load()
print("Paged loaded", len(documents))
#--------------------------------------------------------------------
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk=splitter.split_documents(documents)
print("Chunks created:",len(chunk))
#----------------------------------------------------------------- 
embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')


C:\Users\a810557\AppData\Local\Temp\ipykernel_28224\128217630.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paged loaded 11
Chunks created: 43


In [3]:
vector_store=Chroma.from_documents(documents=chunk,embedding=embedding_model)
retriever=vector_store.as_retriever(search_kwargs={"k":3})

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.4)
prompt=ChatPromptTemplate.from_template("""
You are a researher expert,Answer the user's question using only the information provided in the context.
If the answer is not available in the context, say "I don't know based on the provided document."

Context:
{context}
Question:
{question}

Answer:
    """)

#======================================== 

question='could you please tell what is "Attention all you need" is all about? Please reply in bullet points'
context=retriever.invoke(question)

final=prompt.invoke({'question':question, 'context':context})

ans=llm.invoke(final)
print(ans.content)


'''def ask_rag(question):
retrieved_docs= retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs)
 

    messages=prompt.invoke({"context":context, "question":question})
    response = llm.invoke(messages)
    return response.content

while True:
    question=input("Ask question")
    answer=ask_rag(question)
    print(answer)'''


- "Attention Is All You Need" proposes a new network architecture called the Transformer, which is based solely on attention mechanisms.
- It eliminates the need for recurrent and convolutional neural networks, simplifying the model architecture.
- The paper presents experiments on two machine translation tasks, demonstrating that the Transformer models outperform traditional models in quality.
- The proposed models are more parallelizable and require significantly less time to train compared to existing methods.
- The single model with 165 million parameters achieves a BLEU score of 27.5 on English-to-German translation, improving the previous best ensemble result by over 1 BLEU.
- On English-to-French translation, it achieves a BLEU score of 41.1, outperforming the previous single state-of-the-art model by 0.7 BLEU.


'def ask_rag(question):\nretrieved_docs= retriever.invoke(question)\n    context = "\n\n".join(\n        doc.page_content\n        for doc in retrieved_docs)\n\n\n    messages=prompt.invoke({"context":context, "question":question})\n    response = llm.invoke(messages)\n    return response.content\n\nwhile True:\n    question=input("Ask question")\n    answer=ask_rag(question)\n    print(answer)'